# Vector Search Index

In [0]:
%pip install -q databricks-vectorsearch
%restart_python

In [0]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("silver_schema", "f1_rag_silver")
dbutils.widgets.text("reader_group", "account users")

In [0]:
catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
reader_group = dbutils.widgets.get("reader_group")

In [0]:
CHUNKS_TABLE = f"{catalog}.{silver_schema}.wiki_chunks"
INDEX_NAME = f"{catalog}.{silver_schema}.wiki_chunks_index"

In [0]:
import os
import sys
import time

sys.path.append(os.path.abspath(".."))

from databricks.vector_search.client import VectorSearchClient
from rag.config import EMBEDDING_MODEL, VS_ENDPOINT

vsc = VectorSearchClient(disable_notice=True)

### Endpoint

In [0]:
existing_endpoints = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]

if VS_ENDPOINT not in existing_endpoints:
    vsc.create_endpoint_and_wait(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print(f"Endpoint {VS_ENDPOINT} created")
else:
    print(f"Endpoint {VS_ENDPOINT} already exists")

print(vsc.get_endpoint(VS_ENDPOINT)["endpoint_status"]["state"])

### Index

In [0]:
existing_indexes = [i["name"] for i in vsc.list_indexes(VS_ENDPOINT).get("vector_indexes", [])]

if INDEX_NAME not in existing_indexes:
    index = vsc.create_delta_sync_index_and_wait(
        endpoint_name=VS_ENDPOINT,
        index_name=INDEX_NAME,
        source_table_name=CHUNKS_TABLE,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk_text",
        embedding_model_endpoint_name=EMBEDDING_MODEL,
        columns_to_sync=["chunk_id", "chunk_size", "page_id", "title", "category",
                         "section", "chunk_text", "url", "revision_id"],
    )
    print(f"Index {INDEX_NAME} created")
else:
    index = vsc.get_index(endpoint_name=VS_ENDPOINT, index_name=INDEX_NAME)
    index.sync()  # pick up changes in wiki_chunks since the last sync
    print(f"Index {INDEX_NAME} exists, sync started")

In [0]:
while True:
    status = index.describe()["status"]
    if status.get("ready") and "SYNCING" not in status.get("detailed_state", ""):
        break
    print(f"{status.get('detailed_state')} — waiting...")
    time.sleep(20)

indexed_rows = status.get("indexed_row_count")
table_rows = spark.table(CHUNKS_TABLE).count()
print(f"State: {status['detailed_state']}")
print(f"Indexed rows: {indexed_rows}, rows in {CHUNKS_TABLE}: {table_rows}")

### Access

In [0]:
spark.sql(f"GRANT SELECT ON TABLE {INDEX_NAME} TO `{reader_group}`")
display(spark.sql(f"SHOW GRANTS ON TABLE {INDEX_NAME}"))